<a href="https://colab.research.google.com/github/kafSaugat7/NLP/blob/main/Fake_News_Detection_Bidirectional_LSTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
df=pd.read_csv('/content/drive/MyDrive/NLP/train.csv')

In [ ]:
df.head()

,id,title,author,text,label
0,0,House Dem Aide: We Didn’t Even See Comey’s Let...,Darrell Lucus,House Dem Aide: We Didn’t Even See Comey’s Let...,1
1,1,"FLYNN: Hillary Clinton, Big Woman on Campus - ...",Daniel J. Flynn,Ever get the feeling your life circles the rou...,0
2,2,Why the Truth Might Get You Fired,Consortiumnews.com,"Why the Truth Might Get You Fired October 29, ...",1
3,3,15 Civilians Killed In Single US Airstrike Hav...,Jessica Purkiss,Videos 15 Civilians Killed In Single US Airstr...,1
4,4,Iranian woman jailed for fictional unpublished...,Howard Portnoy,Print \nAn Iranian woman has been sentenced to...,1


In [ ]:
df.isnull().sum()

,0
id,0
title,558
author,1957
text,39
label,0


In [ ]:
df=df.dropna()

In [ ]:
df.isnull().sum()


,0
id,0
title,0
author,0
text,0
label,0


In [ ]:
df.shape

(18285, 5)

In [ ]:
#get the independent features
X=df.drop('label',axis=1)

In [ ]:
y=df['label']

In [ ]:
#check whether dataset is balaned or not
y.value_counts()

,count
label,
0,10361
1,7924


In [ ]:
import tensorflow as tf

In [ ]:
from tensorflow.keras.layers import Embedding
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.text import one_hot
from tensorflow.keras.layers import LSTM
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Bidirectional


In [ ]:
#vocabulary size
voc_size=5000

In [ ]:
#make copy of x
message=X.copy()

In [ ]:
message.head()

,id,title,author,text
0,0,House Dem Aide: We Didn’t Even See Comey’s Let...,Darrell Lucus,House Dem Aide: We Didn’t Even See Comey’s Let...
1,1,"FLYNN: Hillary Clinton, Big Woman on Campus - ...",Daniel J. Flynn,Ever get the feeling your life circles the rou...
2,2,Why the Truth Might Get You Fired,Consortiumnews.com,"Why the Truth Might Get You Fired October 29, ..."
3,3,15 Civilians Killed In Single US Airstrike Hav...,Jessica Purkiss,Videos 15 Civilians Killed In Single US Airstr...
4,4,Iranian woman jailed for fictional unpublished...,Howard Portnoy,Print \nAn Iranian woman has been sentenced to...


In [ ]:
import nltk
import re
from nltk.corpus import stopwords


In [ ]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [ ]:
message.reset_index(inplace=True)

In [ ]:
#Dataset Preprocessing
#Dataset Preprocessing
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()
corpus = []
for i in range(0, len(message)):
    # Access the 'title' and 'author' columns using correct column names
    reviews = re.sub('[^&a-zA-Z]', ' ', message['title'][i])
    reviews = reviews.lower()
    reviews = reviews.split()
    reviews = [ps.stem(word) for word in reviews if not word in stopwords.words('english')]
    reviews = ' '.join(reviews)
    corpus.append(reviews)


In [ ]:
corpus[1]


'flynn hillari clinton big woman campu breitbart'

In [ ]:
corpus

['hous dem aid even see comey letter jason chaffetz tweet',
 'flynn hillari clinton big woman campu breitbart',
 'truth might get fire',
 'civilian kill singl us airstrik identifi',
 'iranian woman jail fiction unpublish stori woman stone death adulteri',
 'jacki mason hollywood would love trump bomb north korea lack tran bathroom exclus video breitbart',
 'beno hamon win french socialist parti presidenti nomin new york time',
 'back channel plan ukrain russia courtesi trump associ new york time',
 'obama organ action partner soro link indivis disrupt trump agenda',
 'bbc comedi sketch real housew isi caus outrag',
 'russian research discov secret nazi militari base treasur hunter arctic photo',
 'us offici see link trump russia',
 'ye paid govern troll social media blog forum websit',
 'major leagu soccer argentin find home success new york time',
 'well fargo chief abruptli step new york time',
 'anonym donor pay million releas everyon arrest dakota access pipelin',
 'fbi close hilla

In [ ]:
onehot_repr=[one_hot(words,voc_size)for words in corpus]
onehot_repr

[[2809, 2594, 3320, 2211, 392, 2264, 4891, 1615, 2592, 2248],
 [2583, 2300, 3900, 4137, 1272, 4391, 3827],
 [2586, 1058, 1410, 467],
 [3819, 403, 4959, 3408, 651, 1592],
 [3412, 1272, 4833, 1149, 1323, 936, 1272, 3581, 4056, 3654],
 [1718,
  736,
  3355,
  2162,
  1033,
  4917,
  2519,
  746,
  3477,
  4489,
  3639,
  158,
  999,
  3448,
  3827],
 [1221, 1795, 2326, 1080, 1147, 1611, 1389, 342, 4460, 2565, 2283],
 [4129, 2674, 4818, 2882, 1257, 4152, 4917, 1847, 4460, 2565, 2283],
 [509, 148, 582, 2313, 3008, 378, 2329, 4631, 4917, 3787],
 [4690, 2940, 2248, 319, 1351, 554, 4409, 3286],
 [3789, 3147, 1742, 3489, 4736, 3031, 497, 436, 1197, 4052, 1018],
 [3408, 1349, 392, 378, 4917, 1257],
 [4034, 340, 1784, 550, 3607, 148, 179, 1378, 3146],
 [1796, 3677, 2353, 836, 1638, 4100, 3418, 4460, 2565, 2283],
 [1337, 3264, 1351, 2647, 4928, 4460, 2565, 2283],
 [3966, 1947, 2712, 3413, 4738, 1093, 4187, 3042, 3656, 1619],
 [2418, 2367, 2300],
 [4001, 924, 248, 3034, 4917, 4654, 2220, 3827],
 [3

In [ ]:
onehot_repr[1]

[2583, 2300, 3900, 4137, 1272, 4391, 3827]

**Embedding Representation**

In [ ]:

sent_length=20
embedded_docs = pad_sequences(onehot_repr,padding='pre',maxlen=sent_length) # Assign the result of pad_sequences to embedded_docs
embedded_docs

array([[   0,    0,    0, ..., 1615, 2592, 2248],
       [   0,    0,    0, ..., 1272, 4391, 3827],
       [   0,    0,    0, ..., 1058, 1410,  467],
       ...,
       [   0,    0,    0, ..., 4460, 2565, 2283],
       [   0,    0,    0, ..., 3183,  358, 2703],
       [   0,    0,    0, ..., 2296,  149, 1668]], dtype=int32)

In [ ]:
from logging import logProcesses
#Creating model
embedding_vector_features=40
model=Sequential()
model.add(Embedding(voc_size,embedding_vector_features,input_length=sent_length))
model.add(Bidirectional(LSTM(100)))
model.add(Dense(1,activation='sigmoid'))
model.compile(loss='binary_crossentropy',optimizer='adam',metrics=['accuracy'])
input_shape = (None, sent_length)  # Replace None with your batch size if fixed
model.build(input_shape)
print(model.summary())

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding (Embedding)                │ (None, 20, 40)              │         200,000 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ bidirectional (Bidirectional)        │ (None, 200)                 │         112,800 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 1)                   │             201 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 313,001 (1.19 MB)

 Trainable params: 313,001 (1.19 MB)

 Non-trainable params: 0 (0.00 B)

None


In [ ]:
len(embedded_docs),y.shape

(18285, (18285,))

In [ ]:
import numpy as np
X_final=np.array(embedded_docs)
y_final=np.array(y)

In [ ]:
X_final.shape,y_final.shape

((18285, 20), (18285,))

In [ ]:
# Now proceed with train_test_split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_final, y_final, test_size=0.33, random_state=42)

In [ ]:

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_final, y_final, test_size=0.33, random_state=42)

In [ ]:
#Model Training
model.fit(X_train,y_train,validation_data=(X_test,y_test),epochs=10,batch_size=64)

Epoch 1/10
192/192 ━━━━━━━━━━━━━━━━━━━━ 18s 70ms/step - accuracy: 0.7757 - loss: 0.4198 - val_accuracy: 0.9191 - val_loss: 0.1966
Epoch 2/10
192/192 ━━━━━━━━━━━━━━━━━━━━ 14s 74ms/step - accuracy: 0.9445 - loss: 0.1376 - val_accuracy: 0.9241 - val_loss: 0.1978
Epoch 3/10
192/192 ━━━━━━━━━━━━━━━━━━━━ 18s 62ms/step - accuracy: 0.9690 - loss: 0.0897 - val_accuracy: 0.9191 - val_loss: 0.2098
Epoch 4/10
192/192 ━━━━━━━━━━━━━━━━━━━━ 22s 68ms/step - accuracy: 0.9818 - loss: 0.0640 - val_accuracy: 0.9114 - val_loss: 0.2618
Epoch 5/10
192/192 ━━━━━━━━━━━━━━━━━━━━ 14s 75ms/step - accuracy: 0.9842 - loss: 0.0491 - val_accuracy: 0.9145 - val_loss: 0.3467
Epoch 6/10
192/192 ━━━━━━━━━━━━━━━━━━━━ 13s 70ms/step - accuracy: 0.9904 - loss: 0.0284 - val_accuracy: 0.9120 - val_loss: 0.3766
Epoch 7/10
192/192 ━━━━━━━━━━━━━━━━━━━━ 14s 73ms/step - accuracy: 0.9934 - loss: 0.0231 - val_accuracy: 0.9143 - val_loss: 0.3894
Epoch 8/10
192/192 ━━━━━━━━━━━━━━━━━━━━ 20s 71ms/step - accuracy: 0.9957 - loss: 0.0191 - 

In [ ]:
y_pred=model.predict(X_test)

189/189 ━━━━━━━━━━━━━━━━━━━━ 3s 16ms/step


In [ ]:
y_pred=np.where(y_pred > 0.5, 1, 0)

In [ ]:
y_pred

array([[1],
       [0],
       [0],
       ...,
       [0],
       [1],
       [0]])

In [ ]:
from sklearn.metrics import confusion_matrix

In [ ]:
confusion_matrix(y_test,y_pred)

array([[3140,  279],
       [ 288, 2328]])

In [ ]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test,y_pred)

0.9060480530240265

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.92      0.92      0.92      3419
           1       0.89      0.89      0.89      2616

    accuracy                           0.91      6035
   macro avg       0.90      0.90      0.90      6035
weighted avg       0.91      0.91      0.91      6035



In [ ]:
def predict_fake_news(news_article, model, voc_size=5000, sent_length=20):
    review = re.sub('[^a-zA-Z]', ' ', news_article)  # Clean the text
    review = review.lower()  # Convert to lowercase
    review = review.split()  # Split into words
    review = [ps.stem(word) for word in review if word not in stopwords.words('english')]
    review = ' '.join(review)  # Join words back into a string
    onehot_repr = one_hot(review, voc_size)  # One-hot encoding
    embedded_doc = pad_sequences([onehot_repr], padding='pre', maxlen=sent_length)  # Padding
    prediction = model.predict(embedded_doc)  # Model prediction
    return "Fake News" if prediction > 0.5 else "Not Fake News"

# Example prediction on a new article
sample_article = "flynn hillari clinton big woman campu breitbart"
result = predict_fake_news(sample_article, model)
print("Prediction for the article:", result)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
Prediction for the article: Not Fake News


In [ ]:
# Example prediction on a new article
sample_article = "hous dem aid even see comey letter jason chaffetz tweet"
result = predict_fake_news(sample_article, model)
print("Prediction for the article:", result)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
Prediction for the article: Fake News


In [ ]:
#leakage problem hudaina due to train test done in better way